In [44]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, Perceptron
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegressionCV
import matplotlib.pyplot as plt

# CMSE 381 Final Project Template

**INSTRUCTIONS**: This is a template to help organize your project.  All projects should include the 5 major sections below (you do not need to use this template file).  If you use this file, complete your work below and remove content in parentheses. Also, remove this current cell.  

#### CMSE 381 Final Project
### &#9989; Group members: NAME1, NAME2
### &#9989; Section_002
#### &#9989; 04/22/26

# Assessing Drivers of Student Success

## Background and Motivation

Understanding which student-level factors drive academic performance has clear implications for how schools allocate resources and how students prioritize their time. While study habits, attendance, and home environment are well-established correlates of achievement, their relative importance, their interactions, and the extent to which they depend on institutional context remain open empirically.

This project uses the Student Performance Factors dataset (6,607 students, 19 behavioral, background, and institutional features) to investigate four questions:

- Which student-level factors predict Exam_Score, and how much of its variance can they explain?
- Which features survive L1 regularization, i.e., which are genuinely informative vs. redundant?
- Do nonlinear effects and feature interactions meaningfully improve predictions?
- Can a model reliably distinguish Public from Private school students from the same features? If not, institutional type likely adds little beyond individual-level signal.

## Methodology

### Data
_(Describe the data you are using. What variables are you using? What they mean? Why did you choose them?)_

We use the **Student Performance Factors** dataset: 6,607 student records and 20 columns (19 predictors + the `Exam_Score` target). The dataset is self-contained — no joins, no external lookups — which keeps the focus on modeling rather than data engineering. A small number of categorical entries are missing (~235 total cells), handled implicitly by `pd.get_dummies`.

The variables fall into three groups:

- **Academic behavior (numeric):** `Hours_Studied`, `Attendance`, `Previous_Scores`, `Tutoring_Sessions` — direct measures of effort and prior achievement, and the features most plausibly causal for exam outcomes.
- **Lifestyle (numeric):** `Sleep_Hours`, `Physical_Activity` — included to test whether non-academic routines carry independent signal once effort is controlled for.
- **Background and context (categorical):** `Parental_Involvement`, `Access_to_Resources`, `Family_Income`, `Parental_Education_Level`, `Teacher_Quality`, `Peer_Influence`, `Motivation_Level`, `Learning_Disabilities`, `Internet_Access`, `Extracurricular_Activities`, `Distance_from_Home`, `Gender`, and `School_Type` — ordinal and nominal features capturing the student's environment. These are one-hot encoded (with `drop_first=True` to avoid multicollinearity).

Two targets are used:

- `Exam_Score` (continuous, range 55–101) — the regression target.
- `School_Type` (Public vs. Private, ~70/30 split) — the classification target. Because the class balance is uneven, stratified splits and stratified k-fold CV are used throughout the classification experiments.

All 19 predictors are retained rather than hand-pruned up front: feature selection is delegated to L1 regularization, which is one of the questions the project is designed to answer.

In [45]:
# Shared preprocessing — defined once, reused by every model below.
df = pd.read_csv("student_performance_factors.csv")

numeric_cols = [
    "Hours_Studied", "Attendance", "Sleep_Hours",
    "Previous_Scores", "Tutoring_Sessions", "Physical_Activity",
]
categorical_cols = [
    "Parental_Involvement", "Access_to_Resources", "Extracurricular_Activities",
    "Motivation_Level", "Internet_Access", "Family_Income", "Teacher_Quality",
    "Peer_Influence", "Learning_Disabilities", "Parental_Education_Level",
    "Distance_from_Home", "Gender",
]

# Full predictor matrix: numerics kept as-is, categoricals one-hot encoded.
# School_Type is deliberately excluded — it is the classification target.
X_full = pd.concat(
    [df[numeric_cols], pd.get_dummies(df[categorical_cols], drop_first=True)],
    axis=1,
)
y_exam = df["Exam_Score"]
y_school = LabelEncoder().fit_transform(df["School_Type"])

In [46]:
print(f"Shape: {df.shape}")
print(f"Missing values (total): {df.isna().sum().sum()}")
print(f"\nSchool_Type class balance:\n{df['School_Type'].value_counts(normalize=True).round(3)}")
df.describe()

Shape: (6607, 20)
Missing values (total): 235

School_Type class balance:
School_Type
Public     0.696
Private    0.304
Name: proportion, dtype: float64


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


### Models for classification _(if applicable)_
_(What models will you be using for classification? Why did you choose to use them? What questions would you answer with them? How would you evaluate if each model? What cross-validation method did you use?)_

In [47]:
# Perceptron baseline for School_Type classification.
# Stratified split preserves the ~70/30 Public/Private ratio.
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_school, test_size=0.2, stratify=y_school, random_state=42,
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

perceptron = Perceptron(random_state=42).fit(X_train_s, y_train)
print(f"Perceptron train acc: {perceptron.score(X_train_s, y_train):.4f}")
print(f"Perceptron test acc:  {perceptron.score(X_test_s, y_test):.4f}")

Perceptron train acc: 0.5682
Perceptron test acc:  0.5772


In [48]:
# Random Forest with stratified 5-fold CV — uses the full encoded feature set,
# so categorical signal (resources, income, etc.) has a chance to contribute.
rf = RandomForestClassifier(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_full, y_school, cv=cv, scoring='accuracy')

print("Fold accuracies:", [f"{s:.4f}" for s in cv_scores])
print(f"Mean CV accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

Fold accuracies: ['0.6906', '0.6884', '0.6995', '0.6934', '0.6858']
Mean CV accuracy: 0.6915 (+/- 0.0047)


In [49]:
# Logistic regression with L1 penalty — CV picks C (inverse regularization
# strength), encouraging a sparse, interpretable classifier. Reuses the
# stratified split and scaler fit above so metrics stay comparable.
logreg_l1 = LogisticRegressionCV(
    penalty='l1', solver='liblinear', Cs=100, cv=5,
    max_iter=10000, random_state=42, scoring='roc_auc',
).fit(X_train_s, y_train)

print(f"Best C: {logreg_l1.C_[0]:.6f}")
print(f"Test accuracy: {logreg_l1.score(X_test_s, y_test):.4f}")

Best C: 0.000100
Test accuracy: 0.5000


### Models for regression _(if applicable)_
_(What models will you be using for regression? Why did you choose to use them? What questions would you answer with them? How would you evaluate if each model? What cross-validation method did you use?)_

In [50]:
# OLS on numeric features — statsmodels gives a full statistical summary
# (coefficients, p-values, diagnostics) in addition to the standard metrics.
X_ols = sm.add_constant(df[numeric_cols])
ols_model = sm.OLS(y_exam, X_ols).fit()
print(ols_model.summary())

y_pred_ols = ols_model.predict(X_ols)
print(f"\nR²:   {r2_score(y_exam, y_pred_ols):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_exam, y_pred_ols)):.4f}")
print(f"MAE:  {mean_absolute_error(y_exam, y_pred_ols):.4f}")

                            OLS Regression Results                            
Dep. Variable:             Exam_Score   R-squared:                       0.598
Model:                            OLS   Adj. R-squared:                  0.598
Method:                 Least Squares   F-statistic:                     1638.
Date:                Wed, 22 Apr 2026   Prob (F-statistic):               0.00
Time:                        12:11:01   Log-Likelihood:                -15338.
No. Observations:                6607   AIC:                         3.069e+04
Df Residuals:                    6600   BIC:                         3.074e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                40.9271      0.33

In [51]:
# LassoCV on the same numeric features — the L1 penalty shrinks low-signal
# coefficients toward zero, and CV picks alpha by minimizing held-out MSE.
X_lasso = StandardScaler().fit_transform(df[numeric_cols])
lasso = LassoCV(cv=5, random_state=42).fit(X_lasso, y_exam)

print(f"Best alpha (CV): {lasso.alpha_:.6f}\n")
print("Lasso coefficients:")
for name, coef in zip(numeric_cols, lasso.coef_):
    print(f"  {name:<20}: {coef:+.4f}")

y_pred_lasso = lasso.predict(X_lasso)
print(f"\nR²:   {r2_score(y_exam, y_pred_lasso):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_exam, y_pred_lasso)):.4f}")
print(f"MAE:  {mean_absolute_error(y_exam, y_pred_lasso):.4f}")

Best alpha (CV): 0.003950

Lasso coefficients:
  Hours_Studied       : +1.7426
  Attendance          : +2.2819
  Sleep_Hours         : -0.0227
  Previous_Scores     : +0.6890
  Tutoring_Sessions   : +0.6034
  Physical_Activity   : +0.1445

R²:   0.5982
RMSE: 2.4658
MAE:  1.3125


In [52]:
# Polynomial Lasso — degree-2 expansion of the full feature set, with L1
# regularization handling the dimensionality blowup. Only interactions that
# genuinely reduce CV error survive with nonzero coefficients.
poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X_full)

X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y_exam, test_size=0.2, random_state=42,
)
scaler_poly = StandardScaler()
X_train_scaled = scaler_poly.fit_transform(X_train)
X_test_scaled = scaler_poly.transform(X_test)

poly_lasso = LassoCV(cv=5, alphas=100, max_iter=10000, random_state=42)
poly_lasso.fit(X_train_scaled, y_train)
y_pred = poly_lasso.predict(X_test_scaled)

print(f"Best alpha: {poly_lasso.alpha_:.6f}")
print(f"Nonzero coefficients: {np.sum(poly_lasso.coef_ != 0)} / {X_poly.shape[1]}")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")

Best alpha: 0.025189
Nonzero coefficients: 130 / 378
R²:   0.7645
RMSE: 1.8247
MAE:  0.5112


### Other methods used _(if applicable)_

_(If this is a preprocessing step to prepare your data for regression or classification models, you should put this subsection before your explanation for the regression or classification models.)_

_(What method did you use otherwise? Why did you choose to use them? What questions would you answer with them? How would you evaluate the results? What cross-validation method did you use when applicable?)_

In [53]:
# PolynomialFeatures preview — a preprocessing step rather than a model.
# Shown here so the dimensionality change fed into the polynomial Lasso above
# is explicit.
print(f"Original features:    {X_full.shape[1]}")
print(f"Degree-2 expansion:   {PolynomialFeatures(degree=2).fit_transform(X_full).shape[1]}")

Original features:    26
Degree-2 expansion:   378


## Results

_(What did you find when you carried out your methods? Some of your code related to
presenting results/figures/data may be replicated from the methods section or may only be present in
this section. All of the plots that you plan on using for your presentation should be present in this
section)_

### classification results
_(What are you trying to do here?)_

In [54]:
# how did you do it

_(How do you interpret what you see?)_

_(What are you doing next?)_

In [55]:
# how did you do it (etc. etc.)

### regression results
_(What are you trying to do here?)_

In [56]:
# how did you do it

_(How do you interpret what you see?)_

_(What are you doing next?)_

In [57]:
# how did you do it (etc. etc.)

### other results
_(What are you trying to do here?)_

In [58]:
# how did you do it

_(How do you interpret what you see?)_

_(What are you doing next?)_

In [59]:
# how did you do it (etc. etc.)

## Discussion and Conclusion

_(What did you learn from your results? What obstacles did you run into? What would you do differently next time? Clearly provide quantitative answers to your question(s)?  At least one of your questions should be answered with numbers.  That is, it is not sufficient to answer "yes" or "no", but rather to say something quantitative such as variable 1 increased roughly 10% for every 1 year increase in variable 2.)_

### discussion on the classification results

### discussion on the regression results

### discussion on the other results

### conclusion and future steps

## Author contribution

_(Please describe the contribution of each member of group)._

## References

_(List the source(s) for any data and/or literature cited in your project.  Ideally, this should be formatted using a formal citation format (MLA or APA or other, your choice!).   Multiple free online citation generators are available such as <a href="http://www.easybib.com/style">http://www.easybib.com/style</a>. **Important:** if you use **any** code that you find on the internet for your project you **must** cite it or you risk losing most/all of the points for you project.)_